<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/RAG_with_New_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Dependencies**

In [ ]:
!pip install -q huggingface_hub langchain_core langchain_huggingface torch tiktoken datasets "chonkie[semantic]" chonkie chromadb "bitsandbytes>=0.46.1"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**Global imports (used throughout notebook)**

In [ ]:
import os
import json
import re
import pandas as pd
from enum import Enum

## **Import dataset from `HuggingFace`**

In [ ]:
from huggingface_hub import login, whoami
from google.colab import userdata

# Colab Secrets (🔑 in the sidebar)
os.environ["hf_token"] = userdata.get("HF_TOKEN")

login(token=os.environ["hf_token"])
print(whoami())

In [ ]:
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_v3.2"

dataset_dict = load_dataset(repo_id, token=os.environ["hf_token"])

# Dataset constants
TRAIN_DF = dataset_dict["train"].to_pandas()
TEST_DF = dataset_dict["test"].to_pandas()
MASKED_DF = dataset_dict["masked_test"].to_pandas()
DISEASE_LABELS = sorted(TRAIN_DF["target_disease"].unique())

print(f"train cases: {len(TRAIN_DF)} | test cases: {len(MASKED_DF)} | diseases: {len(DISEASE_LABELS)}")

**Helper to parse dataset's SOAP notes**

In [ ]:
def parse_soap_sections(note):
    def render(key):
        values = (" ".join(str(v).split()) for v in (note.get(key) or {}).values() if v)
        return " ".join(values).replace("et al.", "et al")  # stop "et al." from being treated as a sentence ending
    return render("subjective"), render("objective"), render("assessment_and_plan")

## **Chunking Strategies**

In [ ]:
from chonkie import (SemanticChunker, RecursiveChunker, RecursiveRules, RecursiveLevel)

# ---------- Chunkers ----------
def perform_semantic_chunking(text, threshold=0.7, chunk_size=500, similarity_window=3, min_sentences_per_chunk=2, skip_window=0, batch=False):

    semantic_chunker = SemanticChunker(
        threshold=threshold,
        chunk_size=chunk_size,              # tokens
        similarity_window=similarity_window,
        min_sentences_per_chunk=min_sentences_per_chunk,
        skip_window=skip_window,
    )

    if batch:
        return [semantic_chunker.chunk(t) for t in text]    # text is a list of strings
    return semantic_chunker.chunk(text)



def perform_structured_chunking(text, chunk_size=500, batch=False):

    rules = RecursiveRules(levels=[
      RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
      RecursiveLevel(delimiters=["\n"], include_delim="prev"),
      RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    ])

    recursive_chunker = RecursiveChunker(
      tokenizer="character",
      chunk_size=chunk_size,              # characters
      rules=rules,
    )

    if batch:
        return [recursive_chunker.chunk(t) for t in text]    # text is a list of strings
    return recursive_chunker.chunk(text)

## **ChromaDB and Vector Database**

**Build `ChromaDB` connection on-demand**

In [ ]:
import chromadb
client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

def make_records(df=TRAIN_DF, chunk_fn=None):
    records = []
    for row in df.itertuples():
        # Extract soap note
        subjective, objective, plan_and_assignment = parse_soap_sections(row.soap_note)

        # chunk_fn returns one list of chunks per section; flatten them into a single list
        chunks = [c for section in chunk_fn([subjective, objective]) for c in section]

        for chunk_id, chunk in enumerate(chunks):
            chunk_text = chunk.text if hasattr(chunk, "text") else str(chunk)
            records.append({
                "document": chunk_text,
                "metadata": {
                    "case_id": row.case_id,
                    "target_disease": row.target_disease
                },
            })
    return records


def build_collection(name, chunk_fn=None, REBUILD=False):
    if chunk_fn is None:
        raise ValueError("chunk_fn must be provided")

    # REBUILD = False, reuse existing connection
    if not REBUILD:
       try:
          collection = client.get_collection(name)
          print("Resuing existing collection!")
          return collection
       except Exception:
          print("Collection not found or corrupt!")
          return None

    # records: list of {"document": str, "metadata": dict}

    records = make_records(chunk_fn=chunk_fn)

    try:
        client.delete_collection(name)
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"}
    )

    batch = client.get_max_batch_size()
    for start in range(0, len(records), batch):
        part = records[start:start + batch]
        collection.upsert(
            ids=[f"chunk_{start + i}" for i in range(len(part))],
            documents=[r["document"] for r in part],
            metadatas=[r["metadata"] for r in part],
        )

    print(f"{name}: built {collection.count()} chunks")
    return collection

In [ ]:
CHROMA_CONFIG = {"name": "semantic_chunking_soap", "chunk_fn": lambda texts: perform_semantic_chunking(texts, batch=True)}

CHROMA_COLLECTION = build_collection(name=CHROMA_CONFIG["name"], chunk_fn=CHROMA_CONFIG["chunk_fn"], REBUILD=False)

## **RAG System**

**Helper to retrieve chunks from `ChromaDB`**

In [ ]:
SEARCH_CHUNKS = 20                                                     # chunks requested from Chroma per query
NOTE_BY_CASE = dict(zip(TRAIN_DF["case_id"], TRAIN_DF["soap_note"]))   # build once; rebuild if TRAIN_DF changes

class RetrievalMode(Enum):
    CASE = "case"
    DISEASE = "disease"

DEFAULT_RETRIEVAL_MODE = RetrievalMode.CASE

def _retrieve(query, collection=CHROMA_COLLECTION, search_k=SEARCH_CHUNKS, retrieval_mode=DEFAULT_RETRIEVAL_MODE):
    """Query the top `search_k` chunks and collapse them to unique cases (closest chunk per case), closest first."""
    n_available = collection.count()
    if n_available == 0:
        return []

    results = collection.query(
        query_texts=[query],
        n_results=min(search_k, n_available),
        include=["documents", "metadatas", "distances"],
    )

    best_by_case = {}
    for document, metadata, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        case_id, distance = metadata["case_id"], float(distance)
        if case_id not in best_by_case or distance < best_by_case[case_id]["distance"]:
            best_by_case[case_id] = {
                "case_id": case_id,
                "target_disease": metadata["target_disease"],
                "text": document,
                "distance": distance,
            }

    # Retrieval mode determines how output is rendered
    hits = sorted(best_by_case.values(), key=lambda hit: (hit["distance"]))

    return _render_by_case(hits) if retrieval_mode == RetrievalMode.CASE else _render_by_disease(hits)

def _clue(case_id):
    """Diagnosis part of a train case's assessment: everything before the 'Plan & Clinical Outcome' section."""
    _, _, assessment = parse_soap_sections(NOTE_BY_CASE[case_id])
    return re.split(r"\s*Plan\s*(?:&|and)\s*Clinical Outcome\s*:?", assessment, maxsplit=1, flags=re.I)[0].strip()


def _render_by_case(hits, max_cases=5):
    """Mode 'case': one entry per retrieved case, closest first."""
    return [{**hit, "clue": _clue(hit["case_id"])} for hit in hits[:max_cases]]


def _render_by_disease(hits):
    """Mode 'disease': one entry per disease, with its support and the clue of its closest case.

    support = number of distinct retrieved cases with that disease, out of n_cases retrieved.
    Sorted by support (highest first), ties broken by closest distance.
    """
    groups = {}
    for hit in hits:                                   # hits are sorted closest-first
        groups.setdefault(hit["target_disease"], []).append(hit)

    entries = [
        {
            "target_disease": disease,
            "support": len(group),
            "n_cases": len(hits),
            "case_id": group[0]["case_id"],            # closest case of this disease
            "distance": group[0]["distance"],
            "clue": _clue(group[0]["case_id"]),        # clue lookup only for the few winners
        }
        for disease, group in groups.items()
    ]
    return sorted(entries, key=lambda e: (-e["support"], e["distance"]))

**Qwen 3 As Small Model (run once!)**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B"

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)
MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
)

**Helper to generate text**

In [ ]:
def _generate(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )

    model_inputs = TOKENIZER([text], return_tensors="pt").to(MODEL.device)

    # conduct text completion
    generated_ids = MODEL.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    content = TOKENIZER.decode(output_ids, skip_special_tokens=True).strip("\n")

    return content

**System prompt (RAG and no RAG)**

In [ ]:
TOP_K_DIAGNOSES = 3

SYS_PROMPT_NO_RAG = (
f"""
You are an expert clinical assistant.\n
You will be given a medical CASE PRESENTATION. You MUST read through this presentation carefully and determine the {TOP_K_DIAGNOSES} most likely diagnoses for this patient.\n

INSTRUCTIONS:\n
1. You MUST output as your answer a numbered list of {TOP_K_DIAGNOSES} unique diseases, from most likely to least likely for the given case presentation.\n
2. You MUST NOT list a disease that the case presentation explicitly rules out (for example, through a negative test or a statement that it was excluded), unless other evidence in the case outweighs that exclusion.\n
3. You MUST respond for each disease with exactly ONE (1) line, formatted as '<rank> <disease>'.\n
4. YOU MUST only respond with diseases from this list, whose names MUST be copied verbatim:\n
"""
+ "\n".join(f"\t- {name}" for name in DISEASE_LABELS)
)

SYS_PROMPT_WITH_RAG = f"""{SYS_PROMPT_NO_RAG}
\nCLUES:
- Use ONLY the retrieved case reports below as evidence (DO NOT diagnose them, only diagnose CASE PRESENTATION):\n"""

**Diagnose (Generation) function**

In [ ]:
def diagnose(question, rag=True):
    if rag:
        # Retrive top-k chunks based on question (query)
        retrieved = _retrieve(question)

        # Format retrieved chunks into usable prompts
        context = "\n\n".join(
            f"POTENTIAL DISEASE: {result['target_disease']}"
            + (f" ({result['support']}/{result['n_cases']} similar cases)" if "support" in result else "")  # Support for disease mode
            + f"\nCLUE: {result['clue']}"
            for result in retrieved
        )

    else:
        retrieved = []
        context = ""

    user_prompt = (
        f"{context}\n\n"
        f"CASE PRESENTATION (TO BE DIAGNOSED):\n{question}"
    )

    # Determine whether to use RAG mode or not
    sys_prompt = SYS_PROMPT_WITH_RAG if rag else SYS_PROMPT_NO_RAG

    # Generate response with chunks' aid
    response = _generate(system_prompt=sys_prompt, user_prompt=user_prompt)

    return response, retrieved

## **Benchmarks and metrics (Evaluation)**

**Evaluate retrieval**

In [ ]:
def evaluate_retrieval_by_disease(eval_df, k=TOP_K_DIAGNOSES):
    """Test the retrieval stage independently, overall and per disease.

    Returns
    -------
    overall    : dict      micro-average over all cases + macro-average over diseases
    by_disease : DataFrame one row per disease (n, recall@k, precision@k, top1_accuracy, mrr@k)
    cases      : DataFrame one row per test case, incl. the retrieved labels (for error analysis)
    """
    records = []

    for row in eval_df.itertuples():
        # Extract soap note and form query (no assement to prevent disease leakage)
        subjective, objective, assessment_and_plan = parse_soap_sections(row.soap_note)
        query = f"""SUBJECTIVE ASSESSMENT\n{subjective}\n\nOBJECTIVE ASSESSMENT\n{objective}\n"""

        # Retrieval
        retrieved = [result["target_disease"] for result in _retrieve(query=query)]

        matches = [_ == row.target_disease for _ in retrieved]
        rank = matches.index(True) + 1 if any(matches) else None

        records.append({
            "case_id": getattr(row, "case_id", row.Index),
            "disease": row.target_disease,
            "hit": any(matches),                                   # recall@k (per case)
            "top1": bool(matches) and matches[0],                  # top-1 accuracy (per case)
            "precision": sum(matches) / len(matches) if matches else 0.0,
            "rr": 1 / rank if rank else 0.0,                       # reciprocal rank
            "retrieved": retrieved,
        })

    cases = pd.DataFrame(records)

    by_disease = (
        cases.groupby("disease")
        .agg(**{
            "n": ("hit", "size"),
            f"recall@{k}": ("hit", "mean"),
            f"precision@{k}": ("precision", "mean"),
            "top1_accuracy": ("top1", "mean"),
            f"mrr@{k}": ("rr", "mean"),
        })
        .sort_values([f"recall@{k}", "top1_accuracy"])             # weakest diseases first
    )

    metric_cols = [c for c in by_disease.columns if c != "n"]
    overall = {
        "k": k,
        "n": len(cases),
        # micro: every case weighted equally
        "micro": {
            f"recall@{k}": cases["hit"].mean(),
            f"precision@{k}": cases["precision"].mean(),
            "top1_accuracy": cases["top1"].mean(),
            f"mrr@{k}": cases["rr"].mean(),
        },
        # macro: every disease weighted equally
        "macro": by_disease[metric_cols].mean().to_dict(),
    }
    return overall, by_disease.round(3), cases


In [ ]:
def sample_train_subset(train_df=TEST_DF, frac=0.10, seed=42, stratify=True):
    """Randomly select ~frac of train_df; stratify=True takes the same fraction per disease (min. 1)."""
    if not stratify:
        return train_df.sample(frac=frac, random_state=seed)
    return pd.concat([
        g.sample(n=max(1, round(frac * len(g))), random_state=seed)
        for _, g in train_df.groupby("target_disease")
    ])

evaluate_retrieval_by_disease(sample_train_subset())

**Set up log folder**

In [ ]:
from datetime import datetime

RAG_EVALUATION_DIR = "/content/drive/MyDrive/colab_projects/VGU_WS26_Project/RAG_Evaluation"

os.makedirs(RAG_EVALUATION_DIR, exist_ok=True)

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
timestamp = "20261128_031833"                       # Custom timestamp

LOG_DIR = os.path.join(RAG_EVALUATION_DIR, timestamp)
os.makedirs(LOG_DIR, exist_ok=True)

# Compare RAG and NO RAG retrieval
RAG_EVAL_JSON = os.path.join(LOG_DIR, f"predictions_rag_{timestamp}.jsonl")
NO_RAG_EVAL_JSON = os.path.join(LOG_DIR, f"predictions_no_rag_{timestamp}.jsonl")

**Evaluate generation (RAG vs. no RAG)**

In [ ]:
from tqdm.auto import tqdm

def run_config(k=SEARCH_CHUNKS, use_rag=True):
    config = {
        "llm": MODEL_NAME,
        "top_k_diagnoses": TOP_K_DIAGNOSES,
        "rag": use_rag,
    }

    if use_rag:
        config.update(
            vector_store="ChromaDB",
            collection_name=CHROMA_CONFIG["name"],
            n_indexed_chunks=CHROMA_COLLECTION.count(),
            retrieve_k=k,
            retrieval_mode=DEFAULT_RETRIEVAL_MODE.name,
        )

    return config

def run_test_predictions(eval_df, path, n_per_disease=None, seed=42, use_rag=True):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    if n_per_disease is None:
        cases = eval_df
    else:
        cases = pd.concat(
            [g.sample(n=min(n_per_disease, len(g)), random_state=seed) for _, g in eval_df.groupby("disease")],
            ignore_index=True,
        )

    # Stored on every record; the chunking function isn't JSON-serializable, so skip it
    config = {key: value for key, value in run_config(use_rag=use_rag).items()}

    print(f"Running {len(cases)} cases -> {path}")

    with open(path, "w", encoding="utf-8") as f:   # "w" overwrites any previous run
        for row in tqdm(cases.itertuples(), total=len(cases), desc="Diagnosing" if use_rag else "Diagnosing (no RAG)"):
            subjective, objective, assessment_and_plan = parse_soap_sections(row.soap_note)
            query = f"""SUBJECTIVE:\n{subjective}\n\nOBJECTIVE:\n{objective}\n"""

            raw_predictions, retrieved = diagnose(question=query, rag=use_rag)
            record = {
                "id": row.case_id,
                "true_disease": row.target_disease,
                "raw_prediction": raw_predictions,
                "config": config,
            }

            if use_rag:
                record.update({
                    "retrieved_ids": [
                        result["case_id"] for result in retrieved
                    ],
                    "retrieved_diseases": [
                        result["target_disease"] for result in retrieved
                    ],
                    "retrieved_distances": [
                        result["distance"] for result in retrieved
                    ],
                })

            f.write(json.dumps(record) + "\n")
            f.flush()   # keep partial results if the run crashes

**Calculate scores (RAG vs no RAG)**

`top1_accuracy`: Fraction of cases where the first diagnosis is correct

`hit_at_k`	   : Fraction where the true disease appears in Gemma's top five diagnoses

`mrr`: Average reciprocal rank of the true disease in Gemma's predictions

`retrieval_recall`: Fraction of cases where Chroma retrieved at least one case with the true disease label

`top1_grounded`:	Fraction where Gemma's first diagnosis label appears among the retrieved disease labels


In [ ]:
def _load_predictions(path):
    if not os.path.exists(path):
        return pd.DataFrame()

    with open(path, encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    return pd.DataFrame(records)

def _parse_diagnoses(prediction, labels=DISEASE_LABELS, top_k=TOP_K_DIAGNOSES):
    """Return up to top_k labels found in `prediction`, ordered by where each first appears
    in the text (not by iterating `labels`) -- so the order reflects the model's own ranking,
    even if it didn't follow the "<rank>. <disease name>" format exactly."""
    prediction_lower = prediction.lower()
    found = [(prediction_lower.find(label.lower()), label) for label in labels]
    found = [(idx, label) for idx, label in found if idx != -1]
    found.sort(key=lambda pair: pair[0])
    return [label for _, label in found][:top_k]

def _score_predictions(predictions_df, top_k=TOP_K_DIAGNOSES):
    # Make a copy of predictions dataframe
    df = predictions_df.copy()

    # RAG-specific fields may be absent in no-RAG prediction files. Append them with NULL as value.
    retrieval_fields = ["retrieved_ids", "retrieved_diseases", "retrieved_chunk_ids", "retrieved_distances"]

    for field in retrieval_fields:
        if field not in df.columns:
            df[field] = [[] for _ in range(len(df))]
        else:
            df[field] = df[field].apply(
                lambda x: x if isinstance(x, list) else []
            )

    # Reformat diseases in "raw_prediction" to valid data structure
    df["predicted_diagnoses"] = df["raw_prediction"].apply(lambda text: _parse_diagnoses(text, top_k=top_k))

    # Introduce evaluation metrics to dataset
    ranks = [
        predicted.index(true) + 1 if true in predicted else None
        for predicted, true in zip(df["predicted_diagnoses"], df["true_disease"])
    ]

    df["top1_correct"] = [bool(p) and p[0] == t for p, t in zip(df["predicted_diagnoses"], df["true_disease"])]
    df["hit_at_k"] = [rank is not None for rank in ranks]
    df["reciprocal_rank"] = [1 / rank if rank else 0.0 for rank in ranks]
    df["true_in_retrieved"] = [t in r for t, r in zip(df["true_disease"], df["retrieved_diseases"])]
    df["top1_grounded"] = [bool(p) and p[0] in r for p, r in zip(df["predicted_diagnoses"], df["retrieved_diseases"])]
    return df


def _summarize(path):
    predictions_df = _load_predictions(path)
    if predictions_df.empty:
        raise FileNotFoundError(f"No predictions in {path} -- run the cell that creates it first.")

    scored_df = _score_predictions(predictions_df)


    retrieved = scored_df[scored_df["true_in_retrieved"]]
    missed = scored_df[~scored_df["true_in_retrieved"]]

    metrics = {
        "n": len(scored_df),
        "top1_accuracy": float(scored_df["top1_correct"].mean()),
        "hit_at_k": float(scored_df["hit_at_k"].mean()),
        "mrr": float(scored_df["reciprocal_rank"].mean()),
        "retrieval_recall": float(scored_df["true_in_retrieved"].mean()),
        "top1_grounded": float(scored_df["top1_grounded"].mean()),
        "hit_at_k_if_retrieved": float(retrieved["hit_at_k"].mean()) if len(retrieved) else float("nan"),
        "hit_at_k_if_not_retrieved": float(missed["hit_at_k"].mean()) if len(missed) else float("nan"),
    }

    return scored_df , metrics

def log_metrics(path, directory=LOG_DIR, timestamp=timestamp, use_rag=True, verbose=True):
    # Evaluate prediction file
    try:
        scored_df, metrics = _summarize(path)
    except FileNotFoundError:
        print("File doesn't exist!")
        return

    # Summary of metrics
    summary = {name: round(value, 3) if isinstance(value, float) else value for name, value in metrics.items()}
    if verbose: print(summary)


    # Prepare merge_df for logging
    merge_df = (
        scored_df[
            ["id", "true_disease", "predicted_diagnoses", "top1_correct", "hit_at_k", "reciprocal_rank", "true_in_retrieved", "top1_grounded"]
        ]
        .merge(
            MASKED_DF[["case_id", "soap_note"]],
            left_on="id",
            right_on="case_id",
            how="left",
        )
        .drop(columns="case_id")   # redundant now that id holds the same values
    )

    rag_text = "rag" if use_rag else "no_rag"

    # Detailed version
    merge_df.to_csv(os.path.join(directory, f"scores_detailed_{timestamp}_{rag_text}.csv"), index=False)

    # Summary version
    with open(os.path.join(directory, f"scores_brief_{timestamp}_{rag_text}.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Saved both files successfully into {timestamp} folder")


**Run LLM and log metrics**

In [ ]:
run_test_predictions(eval_df=MASKED_DF, path=RAG_EVAL_JSON)
run_test_predictions(eval_df=MASKED_DF, path=NO_RAG_EVAL_JSON, use_rag=False)

log_metrics(RAG_EVAL_JSON, use_rag=True)
log_metrics(NO_RAG_EVAL_JSON, use_rag=False)

**Extra metrics**

In [ ]:
def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = successes / n
    denominator = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denominator
    half_width = z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5 / denominator
    return centre - half_width, centre + half_width


def per_disease_report(scored_df):
    rows = []
    for disease, cases in scored_df.groupby("true_disease"):
        n = len(cases)
        hits = int(cases["hit_at_k"].sum())
        ci_low, ci_high = wilson_interval(hits, n)

        answered_first = scored_df[scored_df["predicted_diagnoses"].apply(lambda p: bool(p) and p[0] == disease)]
        rows.append({
            "disease": disease,
            "n": n,
            "top1_rate": cases["top1_correct"].mean(),
            "hit_at_k_rate": hits / n,
            "hit_at_k_ci_low": ci_low,
            "hit_at_k_ci_high": ci_high,
            "retrieval_recall": cases["true_in_retrieved"].mean(),
            "top1_precision": (answered_first["true_disease"] == disease).mean() if len(answered_first) else float("nan"),
        })
    return pd.DataFrame(rows).set_index("disease").sort_values("hit_at_k_rate")


per_disease = per_disease_report(scored_df_rag)
print(
    f"Macro (equal weight per disease) | top-1: {per_disease['top1_rate'].mean():.1%} | "
    f"hit@{TOP_K_DIAGNOSES}: {per_disease['hit_at_k_rate'].mean():.1%} | "
    f"retrieval recall: {per_disease['retrieval_recall'].mean():.1%}"
)
per_disease.round(2)

In [ ]:
import math


def load_scored(path):
    predictions = load_predictions(path)
    if predictions.empty:
        raise FileNotFoundError(f"No predictions in {path} -- run the cell that creates it first.")
    return score_predictions(predictions)


def exact_mcnemar_p(only_rag_right, only_no_rag_right):
    n = only_rag_right + only_no_rag_right
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(only_rag_right, only_no_rag_right) + 1)) / 2**n
    return min(1.0, 2 * tail)


def compare_rag_vs_baseline(rag_scored, base_scored):
    merged = rag_scored.merge(base_scored, on="id", suffixes=("_rag", "_base"))

    rows = []
    for metric in ["top1_correct", "hit_at_k"]:
        rag, base = merged[f"{metric}_rag"], merged[f"{metric}_base"]
        only_rag, only_base = int((rag & ~base).sum()), int((~rag & base).sum())
        rows.append({
            "metric": metric,
            "rag": float(rag.mean()),
            "no_rag": float(base.mean()),
            "difference": float(rag.mean() - base.mean()),
            "only_rag_right": only_rag,
            "only_no_rag_right": only_base,
            "p_value": exact_mcnemar_p(only_rag, only_base),
        })
    mrr_rag, mrr_base = merged["reciprocal_rank_rag"].mean(), merged["reciprocal_rank_base"].mean()
    rows.append({"metric": "mrr", "rag": float(mrr_rag), "no_rag": float(mrr_base), "difference": float(mrr_rag - mrr_base)})
    return merged, pd.DataFrame(rows).set_index("metric")


def per_disease_comparison(merged):
    by_disease = merged.groupby("true_disease_rag")
    comparison = pd.DataFrame({
        "n": by_disease.size(),
        "hit_at_k_rag": by_disease["hit_at_k_rag"].mean(),
        "hit_at_k_no_rag": by_disease["hit_at_k_base"].mean(),
        "top1_rag": by_disease["top1_correct_rag"].mean(),
        "top1_no_rag": by_disease["top1_correct_base"].mean(),
    })
    comparison["hit_at_k_gain"] = comparison["hit_at_k_rag"] - comparison["hit_at_k_no_rag"]
    comparison["top1_gain"] = comparison["top1_rag"] - comparison["top1_no_rag"]
    return comparison.sort_values("top1_gain")


rag_scored = load_scored(RAG_EVAL_JSON)
base_scored = load_scored(NO_RAG_EVAL_JSON)
merged, comparison = compare_rag_vs_baseline(rag_scored, base_scored)

print(f"Compared on {len(merged)} cases that appear in both runs")
display(comparison.round(3))
per_disease_comparison(merged).round(2)
